In [17]:
!pip install plotly ipywidgets


[notice] A new release of pip is available: 25.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import plotly.graph_objs as go
import ipywidgets as widgets
from IPython.display import display
import numpy as np

In [19]:
import vtk
from vtk import *

In [20]:
reader = vtk.vtkXMLImageDataReader()
reader.SetFileName(r"C:\\Users\\durba\\Downloads\\Assignment2_77d416ee-6f81-4276-a8ac-cb0f0a5c41b7\\mixture.vti")
reader.Update()

In [21]:
image_data = reader.GetOutput()
point_data = image_data.GetPointData().GetScalars()
dims = image_data.GetDimensions()
dims

(75, 75, 75)

In [22]:
# https://docs.vtk.org/en/latest/api/python/vtkmodules/vtkmodules.util.numpy_support.html
from vtk.util.numpy_support import vtk_to_numpy
np_arr = vtk_to_numpy(point_data)

In [23]:
np_arr.shape

(421875,)

In [24]:
np_arr[dims[0]-1]

np.float32(-0.19745198)

In [25]:
np_3D_arr = np_arr.reshape(dims[::-1])      ##(z,y,x) format
np_3D_arr[0, 0, dims[0]-1]

np.float32(-0.19745198)

In [26]:
# https://plotly.com/python/3d-isosurface-plots/

z,y,x = np.indices(dims)
modified_df = np_3D_arr.flatten()
def plot_isosurface(isoval):
    fig = go.Figure(data=go.Isosurface(
        x = x.flatten(), 
        y = y.flatten(), 
        z = z.flatten(),
        value = modified_df,
        opacity = 0.7,
        isomin = isoval, 
        isomax = isoval,
        surface_count = 1,
        colorscale = 'plasma',
        caps = dict(x_show=False, y_show=False, z_show=False)
    ))
    fig.update_layout(title='Isosurface Visualization')
    fig.show()

In [27]:
# https://plotly.com/python/histograms/

def plot_histogram(isoval):
    if isoval != 0.0:
        df = modified_df[(modified_df >= (isoval - 0.25)) & (modified_df <= (isoval + 0.25))]
    else:
        df = modified_df  # full dataset

    fig = go.Figure(data=[go.Histogram(x=df, nbinsx=30, marker_color='blue')])
    fig.update_layout(
        title='Histogram',
        xaxis_title='Vortex scalar values',
        yaxis_title='Frequency'
    )
    fig.show()

In [29]:
# isoval slider
def slider():
    isoval_range = (np_3D_arr.min(), np_3D_arr.max())
    
    slider = widgets.FloatSlider(
        value=0.0,
        min=isoval_range[0],
        max=isoval_range[1],
        step=0.01,
        description='Isovalue:'
    )
    output2 = widgets.Output()
    display(slider, output2)
    def on_value_change(change):
        with output2:
            output2.clear_output()
            isoval = change['new']
            # print(isoval)
            plot_isosurface(isoval)
            plot_histogram(isoval)
    
    slider.observe(on_value_change, names='value')

In [31]:
# reset button
from IPython.display import display
button = widgets.Button(description="Reset")
output = widgets.Output()

display(button, output)

def on_button_clicked(b):
    with output:
        slider()
        #reset

button.on_click(on_button_clicked)

Button(description='Reset', style=ButtonStyle())

Output()

In [33]:
slider()

FloatSlider(value=0.0, description='Isovalue:', max=0.43280163407325745, min=-0.9935540556907654, step=0.01)

Output()